# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a guided exploration of the FAIR² clinical dataset using the `mlcroissant` library.

### Dataset Source
This dataset is described by a Croissant schema accessible at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the clinical dataset using `mlcroissant`. This step fetches the schema and establishes programmatic access to records.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load dataset and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title:", getattr(metadata, 'name', 'N/A'))
print("Description:", getattr(metadata, 'description', 'N/A'))

## 2. Data Overview
Explore the available record sets in the dataset and inspect their fields and `@id` values.

You can use the dataset's metadata to browse record set `@id`s, field `@id`s, and column `@id`s for targeted extraction later.

In [ ]:
# List all record sets and their fields/columns with their @ids

record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print("No record sets found in the metadata.")
else:
    for rs in record_sets:
        rs_id = getattr(rs, '@id', None)
        rs_name = getattr(rs, 'name', None)
        print(f"RecordSet @id: {rs_id}")
        print(f"  name: {rs_name}")
        
        fields = getattr(rs, 'field', [])
        if not isinstance(fields, list):
            fields = [fields]
        for field in fields:
            field_id = getattr(field, '@id', None)
            field_name = getattr(field, 'name', None)
            print(f"    Field @id: {field_id}  name: {field_name}")
            columns = getattr(field, 'column', [])
            if not isinstance(columns, list):
                columns = [columns]
            for col in columns:
                col_id = getattr(col, '@id', None)
                col_name = getattr(col, 'name', None)
                print(f"      Column @id: {col_id}  name: {col_name}")
        print('-'*60)

### (Optional) Quick sample: See first record in each available record set
Each row yields a dictionary, keys are field `@id`s.

In [ ]:
# If there are any record sets, show a sample record for each
for record_set in getattr(metadata, 'recordSet', []):
    record_set_id = getattr(record_set, '@id', None)
    print(f"\nSample from RecordSet @id: {record_set_id}")
    try:
        for i, rec in enumerate(dataset.records(record_set=record_set_id)):
            print(rec)
            if i >= 0:  # Just show the first record
                break
    except Exception as e:
        print(f"  Could not load records for this record set: {e}")

## 3. Data Extraction
Load data from all record sets into pandas DataFrames using their `@id` fields. You can then analyze all or specific tables with field and column `@id`s as column names.

In [ ]:
# Collect all record_set @ids
record_set_ids = [getattr(rs, '@id', None) for rs in getattr(metadata, 'recordSet', [])]
dataframes = {}
for rs_id in record_set_ids:
    try:
        rows = list(dataset.records(record_set=rs_id))
        if rows:
            df = pd.DataFrame(rows)
            dataframes[rs_id] = df
            print(f"RecordSet @id: {rs_id} loaded, shape: {df.shape}")
        else:
            print(f"RecordSet @id: {rs_id} yielded no records.")
    except Exception as e:
        print(f"Error loading RecordSet @id: {rs_id}: {e}")

# For demonstration, select the first available record set
selected_rs_id = record_set_ids[0] if record_set_ids else None

if selected_rs_id and selected_rs_id in dataframes:
    print(f"\nColumns in DataFrame for RecordSet @id: {selected_rs_id}:")
    print(list(dataframes[selected_rs_id].columns))
    display(dataframes[selected_rs_id].head())
else:
    print("No available dataframes to display.")

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field (by its field `@id`) for simple EDA, and perform basic filtering, normalization, and grouping. All variable and field references are by `@id` in accordance with best practice.

In [ ]:
# Find a numeric field @id (manually set here or from overview above)
# Edit this cell to choose a different numeric field as needed for your use case!

# Example: Suppose the clinical dataset includes a numeric field 'Age' with @id 'cr:Age'
numeric_field_id = None

# Try to intelligently guess a numeric field @id from columns (modify if needed):
if selected_rs_id and selected_rs_id in dataframes:
    df_cols = dataframes[selected_rs_id].columns
    for col in df_cols:
        if 'age' in col.lower():
            numeric_field_id = col
            break
    if not numeric_field_id:
        for col in df_cols:
            # Try to find a column with typical numeric field (@id) names
            if any(word in col.lower() for word in ['years', 'count', 'number', 'interval', 'score']):
                numeric_field_id = col
                break

    # If not found, just pick first column
    if not numeric_field_id:
        numeric_field_id = df_cols[0]

    # For grouping, try to find a plausible categorical field
    group_field_id = None
    for col in df_cols:
        if any(word in col.lower() for word in ['sex', 'gender', 'group', 'type', 'location']):
            group_field_id = col
            break

    df = dataframes[selected_rs_id]

    # Remove non-numeric and missing values for the numeric field
    df_num = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df_num.mean()  # For demonstration, filter above mean
    filtered_df = df[df_num > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean value): {filtered_df.shape[0]} records")

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - df_num.mean()) / df_num.std()
    )
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        # Display group by mean of the normalized field
        grouped_df = (
            filtered_df.groupby(group_field_id)[f"{numeric_field_id}_normalized"].mean().reset_index()
        )
        print(f"Grouped mean of normalized {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No valid data frame loaded for EDA.")

## 5. Visualization
Let's visualize the distribution of our selected numeric field, and the group means if categorical variable is present. Plots use `matplotlib` and `seaborn` for clarity.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_rs_id and selected_rs_id in dataframes and numeric_field_id:
    df = dataframes[selected_rs_id]
    plt.figure(figsize=(8,4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("Could not visualize; required field(s) missing.")

## 6. Conclusion

In this notebook, we demonstrated how to explore and process a clinical FAIR² dataset with the `mlcroissant` library:
- **Viewed record set and field `@id`s** for robust referencing.
- **Loaded and inspected tabular data** using Croissant schema and DataFrames.
- **Conducted basic EDA and visualization** using only field `@id`s for analysis.

This approach ensures your workflow is reproducible, schema-driven, and ready for further FAIR data science. For in-depth analysis, review the complete schema and select fields by their unique `@id`s as shown.